In [1]:
import shutil
from pathlib import Path
from typing import Literal

import instructor
import lancedb
import pandas as pd
from google.generativeai import GenerativeModel
from IPython.display import Markdown
from lancedb.rerankers import ColbertReranker

from dreamai.ai import ModelName
from dreamai.dialog import Dialog
from dreamai.dialog_models import (
    SourcedRAGResponse,
    StepBackQuestions,
    TableDescription,
)
from dreamai.rag import add_to_lance_table, pdf_to_md_docs
from dreamai.utils import resolve_data_path

%load_ext autoreload
%autoreload 2
%reload_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
ask_kid = instructor.from_gemini(
    client=GenerativeModel(model_name=ModelName.GEMINI_FLASH)
)

In [13]:
table_description_dialog = Dialog.load("dreamai/dialogs/table_description_dialog.json")
table_selection_dialog = Dialog.load("dreamai/dialogs/table_selection_dialog.json")
step_back_dialog = Dialog.load("dreamai/dialogs/step_back_dialog.json")
rag_dialog = Dialog.load("dreamai/dialogs/sourced_rag_dialog.json")

In [4]:
LANCE_URI = "lance/security/"
if Path(LANCE_URI).exists():
    shutil.rmtree(LANCE_URI)

In [5]:
lance_db = lancedb.connect(uri=LANCE_URI)
reranker = ColbertReranker("answerdotai/answerai-colbert-small-v1")

In [6]:
files = "/media/hamza/data2//security_small.pdf"
files = "soccer.txt"
table_descriptions = []
for file in resolve_data_path(data_path=files):
    table_name = Path(file).stem
    md_data = pdf_to_md_docs(file_path=file)
    table_description: TableDescription = ask_kid.create(
        response_model=TableDescription,
        **table_description_dialog.gemini_kwargs(
            template_data={"database_name": table_name, "sample_text": md_data.markdown}
        ),  # type: ignore
        validation_context={"names": [t.name for t in table_descriptions]},
    )
    table_descriptions.append(table_description)
    _ = add_to_lance_table(
        db=lance_db, table_name=table_description.name, data=md_data.chunks
    )

/home/hamza/dev/dreamai/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [21]:
query = "explain the offside rule"

In [22]:
table_names = Literal[
    *[table_description.name for table_description in table_descriptions]
    + ["Assistant"]  # type: ignore
]
table_names

typing.Literal['Soccer', 'Assistant']

In [23]:
table_name: str = ask_kid.create(
    response_model=table_names,
    **table_selection_dialog.gemini_kwargs(
        template_data={
            "query": query,
            "database_list": [
                table_description.model_dump_json(indent=2)
                for table_description in table_descriptions
            ],
        }
    ),  # type: ignore
)

In [24]:
table_name

'Soccer'

In [25]:
table = lance_db.open_table(table_name)
table_name

'Soccer'

In [ ]:
step_back_questions: StepBackQuestions = ask_kid.create(
    response_model=StepBackQuestions,
    **step_back_dialog.gemini_kwargs(user=query),  # type: ignore
)
step_back_questions.questions

In [ ]:
limit = 3
res = (
    pd.concat(
        [
            table.search(
                query=question,  # type: ignore
                query_type="hybrid",
            )
            .rerank(reranker=reranker)  # type: ignore
            .limit(limit)
            .to_pandas()
            for question in step_back_questions.questions + [query]  # type: ignore
        ]
    )
    .drop_duplicates("text")
    .sort_values("_relevance_score", ascending=False)
    .reset_index(drop=True)
)

In [ ]:
documents = res["text"].tolist()
Markdown("\n".join(documents))

In [ ]:
response = ask_kid.create(
    response_model=SourcedRAGResponse,
    **rag_dialog.gemini_kwargs(
        template_data={"documents": documents, "user_query": query}
    ),  # type: ignore
    validation_context={"num_documents": len(documents)},
)

In [ ]:
Markdown(str(response))